# Build injected dirty tables from BART changes

Reconstructs `datasets/{table}/injected/{stem}.csv` by applying each
`bart/*_changes.csv` log to the table's `clean.csv` (see `inject.py`). The
`clean.csv` in each `*_2k` dataset is the exact 2000-row sample BART injected
into, so every change row lines up by rowid (1-based, in CSV order).

Each changes file is named `{table}_{etype}_{rate}[_{level}]_changes.csv`
(see `bart/README.md`): `etype` is `e1` (FD-structured) or `e2` (random),
`rate` is the nominal error percentage, and `level` (e1 only) is the
repairability `low`/`med`/`high`. The output keeps the same fields, e.g.
`injected/e1_r05_high.csv`, `injected/e2_r20.csv`.

BART emitted these runs with unique changes on, so each row is a distinct dirty
cell (one row per cell) and `applied_cells` equals the raw row count — nothing is
dropped. The build still dedupes `(row, attr)` defensively as a safety net.

Jobs are discovered by globbing `bart/{table}_*_changes.csv`, so adding or
removing a changes file changes what gets built — edit `TABLES` to pick tables.

In [ ]:
import sys
from pathlib import Path

import polars as pl

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from inject import apply_changes, parse_changes, read_str

CHANGES = ROOT / "bart"
DATASETS = ROOT / "datasets"

# dataset folders to (re)build; each holds the clean.csv BART injected into.
TABLES = ["hospital_2k", "insurance_claims_2k"]


def discover_jobs(table):
    """Find every bart/{table}_*_changes.csv and derive its output stem.

    File:   {table}_{etype}_{rate}[_{level}]_changes.csv
    Output: {etype}_r{rate:02d}[_{level}]  (under datasets/{table}/injected/)
    """
    jobs = []
    for path in CHANGES.glob(f"{table}_*_changes.csv"):
        tail = path.name.removeprefix(f"{table}_").removesuffix("_changes.csv")
        etype, rate, *rest = tail.split("_")
        level = rest[0] if rest else None
        stem = f"{etype}_r{int(rate):02d}" + (f"_{level}" if level else "")
        jobs.append((path, stem))
    return sorted(jobs, key=lambda j: j[1])

In [ ]:
# Build every injected table; the apply_changes guard fails loudly if a recorded
# clean value disagrees with clean.csv, so a clean run means the data lines up.
stats = []
for table in TABLES:
    clean = read_str(DATASETS / table / "clean.csv")
    out_dir = DATASETS / table / "injected"
    out_dir.mkdir(exist_ok=True)

    for path, stem in discover_jobs(table):
        changes = parse_changes(path)
        deduped = changes.unique(["__row", "attr"], keep="last")  # no-op: BART changes are unique
        total, unique = changes.height, deduped.height

        dirty = apply_changes(clean, deduped)
        dirty.write_csv(out_dir / f"{stem}.csv")

        stats.append(
            {
                "file": path.name,
                "table": table,
                "dirty_cells": unique,
                "dropped": total - unique,
            }
        )
        print(f"{table}/injected/{stem}.csv  ({unique} cells)")

with pl.Config(tbl_rows=len(stats)):
    display(pl.DataFrame(stats))